In [16]:
import os
from pathlib import Path
import uuid
import re

import pandas as pd
import numpy as np
from unidecode import unidecode


In [17]:
# Paths
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TRAINING_PATH = OUTPUT_DIR / "training_data_v2.csv"
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"
MASTER_TEAM_LIST_PATH = Path("master_team_list.csv")

print("📂 DATA_ROOT:", DATA_ROOT.resolve())
print("📂 OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("📄 training_data_v2 path:", TRAINING_PATH)


📂 DATA_ROOT: C:\Python\fpl_pipeline\data
📂 OUTPUT_DIR: C:\Python\fpl_pipeline\output
📄 training_data_v2 path: output\training_data_v2.csv


In [18]:
# %%– Normalization function
def normalize_player_name(name: str) -> str:
    """
    Normalizes player names:
    - to lowercase
    - remove accents
    - strip trailing numbers (e.g. ' 534')
    - remove underscores
    - keep only alphanumeric + spaces
    - collapse multiple spaces
    """
    if pd.isna(name):
        return name

    name = str(name).strip().lower()
    name = unidecode(name)

    # Remove trailing numeric suffixes: "aaron connolly 534", "aaron_connolly_534"
    name = re.sub(r'[\s_]*\d+\s*$', '', name)

    # Replace underscores with spaces
    name = name.replace("_", " ")

    # Keep only alphanumeric + space
    name = "".join(c for c in name if c.isalnum() or c.isspace())

    # Collapse multiple spaces
    name = " ".join(name.split())

    return name


print("🔎 Testing normalize_player_name:")
for t in ["aaron cresswell 376", "Aaron_Connolly534", "Son Heung-Min 123"]:
    print(f"  '{t}' -> '{normalize_player_name(t)}'")


🔎 Testing normalize_player_name:
  'aaron cresswell 376' -> 'aaron cresswell'
  'Aaron_Connolly534' -> 'aaron connolly'
  'Son Heung-Min 123' -> 'son heungmin'


In [ ]:
# %%Φόρτωμα όλων των GW (vaastav gws/)
def load_all_gws(data_root=DATA_ROOT):
    all_seasons = []
    print("\n📥 Loading all GW data from vaastav gws/...")

    for season in sorted(os.listdir(data_root)):
        season_path = data_root / season
        gws_path = season_path / "gws"

        if not season_path.is_dir():
            continue
        if not gws_path.exists():
            # π.χ. 2025-26 που έρχεται από API – το αφήνουμε για άλλη φάση
            print(f"  ⏩ Skipping {season} (no gws/ folder)")
            continue

        gw_files = sorted(
            [f for f in os.listdir(gws_path) if f.startswith("gw") and f.endswith(".csv")],
            key=lambda x: int(x.replace("gw","").replace(".csv",""))
        )

        print(f"  → Season {season}: {len(gw_files)} GWs")

        frames = []
        for fname in gw_files:
            gw = int(fname.replace("gw","").replace(".csv",""))
            df = pd.read_csv(gws_path / fname)

            keep = [
                "name", "element", "minutes", "goals_scored", "assists", "clean_sheets",
                "goals_conceded","yellow_cards","red_cards","total_points",
                "influence","creativity","threat","ict_index",
                "opponent_team","was_home"
            ]
            cols = [c for c in keep if c in df.columns]
            df = df[cols].copy()

            df["season"] = season
            df["Gameweek"] = gw
            frames.append(df)

        all_seasons.append(pd.concat(frames, ignore_index=True))

    df_gws = pd.concat(all_seasons, ignore_index=True)
    print(f"✅ GW rows in df_gws: {len(df_gws):,}")
    
    # Remove 2025-26 temporarily until processed separately
    df_gws = df_gws[df_gws["season"] != "2025-26"].copy()
    
    return df_gws


df_gws = load_all_gws()
display(df_gws.head())
print(df_gws["season"].value_counts().sort_index())



📥 Loading all GW data from vaastav gws/...
  → Season 2018-19: 38 GWs
  → Season 2019-20: 38 GWs
  → Season 2020-21: 38 GWs


C:\Users\SOFI\AppData\Local\Temp\ipykernel_22392\1607694899.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


  → Season 2021-22: 38 GWs
  → Season 2022-23: 38 GWs


C:\Users\SOFI\AppData\Local\Temp\ipykernel_22392\1607694899.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_seasons.append(pd.concat(frames, ignore_index=True))


  → Season 2023-24: 38 GWs
  → Season 2024-25: 38 GWs
  ⏩ Skipping 2025-26 (no gws/ folder)
✅ GW rows in df_gws: 171,993


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,influence,creativity,threat,ict_index,opponent_team,was_home,season,Gameweek
0,Aaron_Cresswell_402,402,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,12,False,2018-19,1
1,Aaron_Lennon_83,83,90,0,0,1,0,0,0,3,10.0,12.3,17.0,3.9,16,False,2018-19,1
2,Aaron_Mooy_199,199,90,0,0,0,3,0,0,2,20.2,18.2,0.0,3.8,6,True,2018-19,1
3,Aaron_Ramsey_14,14,53,0,0,0,1,0,0,1,9.4,10.8,9.0,2.9,13,True,2018-19,1
4,Aaron_Wan-Bissaka_145,145,90,0,1,1,0,0,0,12,46.0,14.0,0.0,6.0,9,False,2018-19,1


season
2018-19    21790
2019-20    16556
2020-21    24365
2021-22    25447
2022-23    26505
2023-24    29725
2024-25    27605
Name: count, dtype: int64


In [20]:
#– Φόρτωμα players_raw ανά σεζόν
# %%
def load_players_raw_by_season():
    players = {}
    print("\n📥 Loading players_raw per season...")

    for season in sorted(df_gws["season"].unique()):
        path = DATA_ROOT / season / "players_raw.csv"
        if not path.exists():
            print(f"  ⚠️ No players_raw for {season}")
            continue

        df = pd.read_csv(path)

        # unify ID col
        if "id" in df.columns:
            df.rename(columns={"id": "element"}, inplace=True)

        keep = ["element","team","element_type","web_name","first_name","second_name"]
        keep = [c for c in keep if c in df.columns]
        df = df[keep].copy()

        df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
        if "team" in df.columns:
            df["team"] = pd.to_numeric(df["team"], errors="coerce").astype("Int64")

        players[season] = df.set_index("element")
        print(f"  → {season}: {len(df)} players in players_raw")

    return players


players_raw_by_season = load_players_raw_by_season()



📥 Loading players_raw per season...
  → 2018-19: 624 players in players_raw
  → 2019-20: 666 players in players_raw
  → 2020-21: 713 players in players_raw
  → 2021-22: 737 players in players_raw
  → 2022-23: 778 players in players_raw
  → 2023-24: 865 players in players_raw
  → 2024-25: 804 players in players_raw


In [21]:
# %% Teams per season (με master_team_list fallback για 2018-19)
master_teams_df = pd.read_csv(MASTER_TEAM_LIST_PATH) if MASTER_TEAM_LIST_PATH.exists() else None
if master_teams_df is not None:
    master_teams_df.columns = [c.lower() for c in master_teams_df.columns]

def load_teams_for_season(season: str) -> pd.DataFrame:
    """
    Επιστρέφει DF με στήλες: Team ID, Team Name, short_name
    - teams.csv αν υπάρχει
    - αλλιώς master_team_list.csv (π.χ. 2018-19)
    """
    path = DATA_ROOT / season / "teams.csv"

    if path.exists():
        df = pd.read_csv(path)
        idcol = "id" if "id" in df.columns else "code"
        df.rename(columns={idcol: "Team ID", "name": "Team Name"}, inplace=True)
        df["Team ID"] = pd.to_numeric(df["Team ID"], errors="coerce").astype("Int64")
        if "short_name" not in df.columns:
            df["short_name"] = df["Team Name"]
        return df[["Team ID", "Team Name", "short_name"]]

    if master_teams_df is not None:
        sub = master_teams_df[master_teams_df["season"] == season].copy()
        if not sub.empty:
            sub.rename(columns={"team": "Team ID", "team_name": "Team Name"}, inplace=True)
            sub["Team ID"] = pd.to_numeric(sub["Team ID"], errors="coerce").astype("Int64")
            sub["short_name"] = sub["Team Name"]
            print(f"  🔁 Using master_team_list for {season}")
            return sub[["Team ID", "Team Name", "short_name"]]

    raise RuntimeError(f"❌ No team info for {season}")


In [22]:
# %% – Φόρτωμα fixtures σε “long” μορφή (home/away rows)
def load_fixtures_long():
    rows = []
    print("\n📥 Building fixture_long from fixtures.csv ...")

    for season in sorted(df_gws["season"].unique()):
        fx_path = DATA_ROOT / season / "fixtures.csv"
        if not fx_path.exists():
            print(f"  ⚠️ No fixtures.csv for {season}, skipping.")
            continue

        fx = pd.read_csv(fx_path)
        if "event" in fx.columns:
            fx["Gameweek"] = fx["event"]
        elif "round" in fx.columns:
            fx["Gameweek"] = fx["round"]
        else:
            raise RuntimeError(f"No event/round column in fixtures for {season}")

        for _, r in fx.iterrows():
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_h"),
                "Opponent ID": r.get("team_a"),
                "Is Home": True,
                "Difficulty": r.get("team_h_difficulty", np.nan)
            })
            rows.append({
                "season": season,
                "Gameweek": r["Gameweek"],
                "Team ID": r.get("team_a"),
                "Opponent ID": r.get("team_h"),
                "Is Home": False,
                "Difficulty": r.get("team_a_difficulty", np.nan)
            })

    fixture_long = pd.DataFrame(rows)
    fixture_long["Gameweek"] = pd.to_numeric(fixture_long["Gameweek"], errors="coerce").astype("Int64")
    fixture_long["Team ID"] = pd.to_numeric(fixture_long["Team ID"], errors="coerce").astype("Int64")
    fixture_long["Opponent ID"] = pd.to_numeric(fixture_long["Opponent ID"], errors="coerce").astype("Int64")
    fixture_long["Is Home"] = fixture_long["Is Home"].astype(bool)

    print("✅ fixture_long rows:", len(fixture_long))
    return fixture_long


fixture_long = load_fixtures_long()
display(fixture_long.head())



📥 Building fixture_long from fixtures.csv ...
✅ fixture_long rows: 5320


,season,Gameweek,Team ID,Opponent ID,Is Home,Difficulty
0,2018-19,1,14,11,True,3
1,2018-19,1,11,14,False,4
2,2018-19,1,15,17,True,4
3,2018-19,1,17,15,False,3
4,2018-19,1,2,5,True,2


In [23]:
# %% 
# Build df_core per season by deriving Player Team ID from fixtures (perfect fix)
print("\n📦 Building df_core per season using fixture-based team identification...")

def build_season_core(season: str, df_season: pd.DataFrame) -> pd.DataFrame:
    print(f"\n🧱 Season {season}...")

    df = df_season.copy()
    df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
    df["Gameweek"] = pd.to_numeric(df["Gameweek"], errors="coerce").astype("Int64")
    df["opponent_team"] = pd.to_numeric(df["opponent_team"], errors="coerce").astype("Int64")

    # players_raw for names, element_type ONLY (NOT team)
    pr = players_raw_by_season.get(season)
    df = df.merge(
        pr[["element_type", "web_name", "first_name", "second_name"]],
        left_on="element", right_index=True, how="left"
    )

    # fixtures for true team mapping
    fx_path = DATA_ROOT / season / "fixtures.csv"
    fx = pd.read_csv(fx_path)
    fx["Gameweek"] = pd.to_numeric(
        fx["event"] if "event" in fx.columns else fx["round"], 
        errors="coerce"
    ).astype("Int64")

    # reconstruct Player Team ID + Opponent Difficulty by matching opponent_team
    rows = []
    for _, r in fx.iterrows():
        gw = int(r["Gameweek"])
        h = int(r["team_h"])
        a = int(r["team_a"])
        dh = int(r["team_h_difficulty"])
        da = int(r["team_a_difficulty"])

        rows.append({"Gameweek": gw, "OppKey": a, "Player Team ID": h, "Opponent ID": a, "Is Home": True, "Opponent Difficulty": dh})
        rows.append({"Gameweek": gw, "OppKey": h, "Player Team ID": a, "Opponent ID": h, "Is Home": False, "Opponent Difficulty": da})

    opp_map = pd.DataFrame(rows)
    opp_map["Gameweek"] = opp_map["Gameweek"].astype("Int64")
    opp_map["OppKey"] = opp_map["OppKey"].astype("Int64")

    df = df.merge(
        opp_map,
        left_on=["Gameweek", "opponent_team"],
        right_on=["Gameweek", "OppKey"],
        how="left"
    ).drop(columns=["OppKey"])

    # team names
    teams_df = load_teams_for_season(season)

    df = df.merge(
        teams_df[["Team ID", "Team Name"]],
        left_on="Player Team ID",
        right_on="Team ID",
        how="left"
    ).rename(columns={"Team Name": "Player Team Name"}).drop(columns=["Team ID"])

    df = df.merge(
        teams_df.rename(columns={"Team ID": "Opponent ID", "Team Name": "Opponent Name"})[["Opponent ID", "Opponent Name"]],
        on="Opponent ID",
        how="left"
    )

    # clean names
    df["Player Name"] = (
        df["first_name"].fillna("") + " " + df["second_name"].fillna("")
    ).str.strip().replace("", np.nan).fillna(df["name"])
    df["Web Name"] = df["web_name"]

    return df


core_frames = [build_season_core(s, df_gws[df_gws["season"] == s]) for s in sorted(df_gws["season"].unique())]
df_core = pd.concat(core_frames, ignore_index=True)

print("\n✅ df_core rebuilt with 100% accurate Player Team IDs")
display(df_core.head())



📦 Building df_core per season using fixture-based team identification...

🧱 Season 2018-19...
  🔁 Using master_team_list for 2018-19

🧱 Season 2019-20...

🧱 Season 2020-21...

🧱 Season 2021-22...

🧱 Season 2022-23...

🧱 Season 2023-24...

🧱 Season 2024-25...

✅ df_core rebuilt with 100% accurate Player Team IDs


,name,element,minutes,goals_scored,assists,clean_sheets,goals_conceded,yellow_cards,red_cards,total_points,...,first_name,second_name,Player Team ID,Opponent ID,Is Home,Opponent Difficulty,Player Team Name,Opponent Name,Player Name,Web Name
0,Aaron_Cresswell_402,402,0,0,0,0,0,0,0,0,...,Aaron,Cresswell,19,12,False,5,West Ham,Liverpool,Aaron Cresswell,Cresswell
1,Aaron_Lennon_83,83,90,0,0,1,0,0,0,3,...,Aaron,Lennon,4,16,False,2,Burnley,Southampton,Aaron Lennon,Lennon
2,Aaron_Mooy_199,199,90,0,0,0,3,0,0,2,...,Aaron,Mooy,10,6,True,4,Huddersfield,Chelsea,Aaron Mooy,Mooy
3,Aaron_Ramsey_14,14,53,0,0,0,1,0,0,1,...,Aaron,Ramsey,1,13,True,4,Arsenal,Man City,Aaron Ramsey,Ramsey
4,Aaron_Wan-Bissaka_145,145,90,0,1,1,0,0,0,12,...,Aaron,Wan-Bissaka,7,9,False,2,Crystal Palace,Fulham,Aaron Wan-Bissaka,Wan-Bissaka


In [24]:
# %%– Diagnostic: Opponent Difficulty completeness
print("🔍 Opponent Difficulty — overall summary (df_core)")
total_rows = len(df_core)
missing_od = df_core["Opponent Difficulty"].isna().sum()
print(f"  Total rows: {total_rows:,}")
print(f"  Missing Opponent Difficulty: {missing_od:,} ({missing_od/total_rows*100:.4f}%)")

print("\n🔍 Missing Opponent Difficulty by season:")
season_stats = (
    df_core
    .groupby("season")["Opponent Difficulty"]
    .apply(lambda s: s.isna().sum())
    .to_frame("Missing_OD")
)
season_stats["Total_Rows"] = df_core.groupby("season")["Opponent Difficulty"].size()
season_stats["Missing_%"] = (season_stats["Missing_OD"] / season_stats["Total_Rows"] * 100).round(4)

display(season_stats)


🔍 Opponent Difficulty — overall summary (df_core)
  Total rows: 186,718
  Missing Opponent Difficulty: 0 (0.0000%)

🔍 Missing Opponent Difficulty by season:


,Missing_OD,Total_Rows,Missing_%
season,,,
2018-19,0,23109,0.0
2019-20,0,16678,0.0
2020-21,0,27397,0.0
2021-22,0,29837,0.0
2022-23,0,29618,0.0
2023-24,0,31707,0.0
2024-25,0,28372,0.0


In [25]:
# %%– Cleaning & canonical columns
POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

rename_map = {
    "element": "Code",
    "minutes": "Minutes Played",
    "goals_scored": "Goals Scored",
    "assists": "Assists",
    "clean_sheets": "Clean Sheet",
    "goals_conceded": "Goals Conceded",
    "yellow_cards": "Yellow Card",
    "red_cards": "Red Cards",
    "total_points": "Total Points",
    "influence": "Influence",
    "creativity": "Creativity",
    "threat": "Threat",
    "ict_index": "ICT Index",
}

df_clean = df_core.copy()
df_clean.rename(columns=rename_map, inplace=True)

# Position από element_type
if "element_type" in df_clean.columns:
    df_clean["Position"] = df_clean["element_type"].map(POS_MAP)
else:
    df_clean["Position"] = np.nan

# Normalized name
df_clean["Player Name Norm"] = df_clean["Player Name"].apply(normalize_player_name)

print("Sample cleaned rows:")
display(df_clean[[
    "Player Name", "Player Name Norm", "Player Team Name",
    "Opponent Name", "Total Points"
]].head())


Sample cleaned rows:


,Player Name,Player Name Norm,Player Team Name,Opponent Name,Total Points
0,Aaron Cresswell,aaron cresswell,West Ham,Liverpool,0
1,Aaron Lennon,aaron lennon,Burnley,Southampton,3
2,Aaron Mooy,aaron mooy,Huddersfield,Chelsea,2
3,Aaron Ramsey,aaron ramsey,Arsenal,Man City,1
4,Aaron Wan-Bissaka,aaron wanbissaka,Crystal Palace,Fulham,12


In [26]:
# %%– Injury flag (3+ συνεχόμενα 0 λεπτά)
df_clean = df_clean.sort_values(["Player Name Norm","season","Gameweek"])
df_clean["Injury/Unavailable"] = 0

for p in df_clean["Player Name Norm"].unique():
    mask = df_clean["Player Name Norm"] == p
    m = df_clean.loc[mask, "Minutes Played"]

    streak = 0
    flags = []
    for x in m:
        if x == 0:
            streak += 1
            flags.append(1 if streak >= 3 else 0)
        else:
            streak = 0
            flags.append(0)
    df_clean.loc[mask, "Injury/Unavailable"] = flags

df_clean[["Player Name", "season", "Gameweek", "Minutes Played", "Injury/Unavailable"]].head(15)


,Player Name,season,Gameweek,Minutes Played,Injury/Unavailable
175884,Aaron Anselmino,2024-25,25,0,0
176695,Aaron Anselmino,2024-25,26,0,0
177478,Aaron Anselmino,2024-25,27,0,1
178266,Aaron Anselmino,2024-25,28,0,1
178924,Aaron Anselmino,2024-25,29,0,1
179696,Aaron Anselmino,2024-25,30,0,1
180488,Aaron Anselmino,2024-25,31,0,1
181481,Aaron Anselmino,2024-25,32,0,1
182705,Aaron Anselmino,2024-25,33,0,1
183431,Aaron Anselmino,2024-25,34,0,1


In [27]:
# %% – Rolling averages L3 & L5
def add_lagged(df):
    metrics = [
        "Total Points", "Minutes Played", "Goals Scored", "Assists",
        "Goals Conceded", "ICT Index", "Threat", "Creativity", "Influence"
    ]

    df = df.sort_values(["Player Name Norm", "season", "Gameweek"])

    for w in [3, 5]:
        for c in metrics:
            if c not in df.columns:
                continue
            new = f"Avg_{c}_L{w}"
            df[new] = (
                df.groupby(["Player Name Norm","season"])[c]
                .transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
                .fillna(0)
            )
    return df


df_lagged = add_lagged(df_clean)
display(df_lagged.head())


,name,Code,Minutes Played,Goals Scored,Assists,Clean Sheet,Goals Conceded,Yellow Card,Red Cards,Total Points,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
175884,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
176695,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
177478,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
178266,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
178924,Aaron Anselmino,774,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
# %%– UUID mapping (χρησιμοποιεί cleaned αν υπάρχει)
if CLEANED_UUID_MAPPING_PATH.exists():
    print("📥 Using cleaned UUID mapping:", CLEANED_UUID_MAPPING_PATH)
    mapping = pd.read_csv(CLEANED_UUID_MAPPING_PATH)

    norm_col = [c for c in mapping.columns if c.lower() == "player name norm"][0]
    uuid_col = [c for c in mapping.columns if c.lower() == "player uuid"][0]

    uuid_map = dict(zip(mapping[norm_col], mapping[uuid_col]))

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(uuid_map)

    print("UUIDs filled:", df_lagged["Player UUID"].notna().sum())

else:
    print("⚠️ No cleaned mapping. Creating NEW stable mapping…")

    unique_norms = sorted(df_lagged["Player Name Norm"].unique())
    new_uuids = [str(uuid.uuid4()) for _ in unique_norms]

    mapping = pd.DataFrame({
        "Player Name Norm": unique_norms,
        "Player UUID": new_uuids
    })

    rep = df_lagged.groupby("Player Name Norm")[["Player Name", "Web Name"]] \
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan).reset_index()

    mapping = mapping.merge(rep, on="Player Name Norm", how="left")

    mapping.to_csv(UUID_MAPPING_PATH, index=False, encoding="utf-8-sig")
    print(f"💾 Saved new UUID mapping → {UUID_MAPPING_PATH}")

    df_lagged["Player UUID"] = df_lagged["Player Name Norm"].map(
        dict(zip(mapping["Player Name Norm"], mapping["Player UUID"]))
    )

display(df_lagged[["Player Name","Player Name Norm","Player UUID"]].head())


📥 Using cleaned UUID mapping: output\player_uuid_mapping_cleaned.csv
UUIDs filled: 185632


,Player Name,Player Name Norm,Player UUID
175884,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
176695,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
177478,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
178266,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815
178924,Aaron Anselmino,aaron anselmino,16b72858-75e4-4125-ad3b-e13f81e6d815


In [29]:
# %%– Τελικό save σε training_data_v2.csv
base_cols = [
    "Player UUID", "Code", "Player Name", "Web Name", "Player Team Name",
    "season", "Gameweek", "Minutes Played", "Goals Scored", "Assists",
    "Clean Sheet", "Goals Conceded", "Yellow Card", "Red Cards",
    "Total Points", "Threat", "ICT Index", "Influence", "Creativity",
    "Opponent Name", "Opponent Difficulty", "Is Home", "Position",
    "Injury/Unavailable"
]

lagged_cols = [c for c in df_lagged.columns if c.startswith("Avg_")]
final_cols = base_cols + lagged_cols

df_final = df_lagged[final_cols].copy()

df_final.to_csv(TRAINING_PATH, index=False, encoding="utf-8-sig")

print("🎉 training_data_v2.csv CREATED!")
print("Rows:", len(df_final))
print("Cols:", len(df_final.columns))
display(df_final.head())


🎉 training_data_v2.csv CREATED!
Rows: 186718
Cols: 42


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
175884,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
176695,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
177478,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
178266,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
178924,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
# %% – Sanity checks
print("Unique seasons:", sorted(df_final["season"].unique()))
print("Max GW per season:", df_final.groupby("season")["Gameweek"].max().to_dict())

print("\nOpponent Difficulty non-null:", df_final["Opponent Difficulty"].notna().sum())
display(
    df_final[
        ["Player Name","Player Team Name","Opponent Name","season","Gameweek","Opponent Difficulty"]
    ].head(20)
)


Unique seasons: ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Max GW per season: {'2018-19': 38, '2019-20': 29, '2020-21': 38, '2021-22': 38, '2022-23': 38, '2023-24': 38, '2024-25': 38}

Opponent Difficulty non-null: 186718


,Player Name,Player Team Name,Opponent Name,season,Gameweek,Opponent Difficulty
175884,Aaron Anselmino,Chelsea,Brighton,2024-25,25,3
176695,Aaron Anselmino,Chelsea,Aston Villa,2024-25,26,4
177478,Aaron Anselmino,Chelsea,Southampton,2024-25,27,1
178266,Aaron Anselmino,Chelsea,Leicester,2024-25,28,1
178924,Aaron Anselmino,Chelsea,Arsenal,2024-25,29,5
179696,Aaron Anselmino,Chelsea,Spurs,2024-25,30,2
180488,Aaron Anselmino,Chelsea,Brentford,2024-25,31,3
181481,Aaron Anselmino,Chelsea,Ipswich,2024-25,32,2
182705,Aaron Anselmino,Chelsea,Fulham,2024-25,33,3
183431,Aaron Anselmino,Chelsea,Everton,2024-25,34,3


In [10]:
print("📂 SAFETY CHECK — Comparing column structure before merge")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")

cols_old = set(df_old.columns)
cols_new = set(df_new.columns)

missing_in_new = cols_old - cols_new
missing_in_old = cols_new - cols_old

print("\n🔍 Columns missing in NEW dataset:", missing_in_new)
print("🔍 Columns missing in OLD dataset:", missing_in_old)



📂 SAFETY CHECK — Comparing column structure before merge
   → Old dataset rows: 186,718
   → New dataset rows: 8,063

🔍 Columns missing in NEW dataset: set()
🔍 Columns missing in OLD dataset: {'Team_Contribution_Rank_Causal', 'Team_Points_Contribution_Causal_Pct', 'Player_Season_Points', 'Team_Points_Contribution_GW_Pct', 'Team_Points_Contribution_Causal', 'Team_Total_Points_GW', 'Team_Total_Points', 'Team_Contribution_Rank_GW', 'Team_Points_Contribution_GW'}


In [5]:
# ================================================================
# SAFE MERGE — KEEP EXACT COLUMN ORDER FROM training_data_v2.csv
# ================================================================

print("📂 Loading old + new datasets with SAFE column order rules...")

OLD_TRAIN = "output/training_data_v2.csv"
NEW_TRAIN = "output/training_data_2025_26.csv"
MERGED_OUT = "output/training_data_v3.csv"

df_old = pd.read_csv(OLD_TRAIN, encoding="utf-8-sig")
df_new = pd.read_csv(NEW_TRAIN, encoding="utf-8-sig")

print(f"   → Old dataset rows: {len(df_old):,}")
print(f"   → New dataset rows: {len(df_new):,}")

# -----------------------------------------
# 1️⃣  Keep only the columns of the OLD file
# (the new dataset will be forced to follow this order)
# -----------------------------------------
old_cols = list(df_old.columns)

# Keep only the columns that exist in both
common_cols = [c for c in old_cols if c in df_new.columns]

print(f"🟡 Keeping {len(common_cols)} common columns in correct order")

df_old_clean = df_old[common_cols]
df_new_clean = df_new[common_cols]

# -----------------------------------------
# 2️⃣ MERGE with correct column order
# -----------------------------------------
df_merged = pd.concat([df_old_clean, df_new_clean], ignore_index=True)

print(f"🎉 MERGED successfully → {len(df_merged):,} rows total")

# -----------------------------------------
# 3️⃣ SAVE FINAL
# -----------------------------------------
df_merged.to_csv(MERGED_OUT, index=False, encoding="utf-8-sig")
print(f"💾 Saved → {MERGED_OUT}")

# -----------------------------------------
# 4️⃣ VERIFY
# -----------------------------------------
print("\n🔍 Sanity check:")
print("Columns:", list(df_merged.columns))


📂 Loading old + new datasets with SAFE column order rules...
   → Old dataset rows: 186,718
   → New dataset rows: 8,063
🟡 Keeping 42 common columns in correct order
🎉 MERGED successfully → 194,781 rows total
💾 Saved → output/training_data_v3.csv

🔍 Sanity check:
Columns: ['Player UUID', 'Code', 'Player Name', 'Web Name', 'Player Team Name', 'season', 'Gameweek', 'Minutes Played', 'Goals Scored', 'Assists', 'Clean Sheet', 'Goals Conceded', 'Yellow Card', 'Red Cards', 'Total Points', 'Threat', 'ICT Index', 'Influence', 'Creativity', 'Opponent Name', 'Opponent Difficulty', 'Is Home', 'Position', 'Injury/Unavailable', 'Avg_Total Points_L3', 'Avg_Minutes Played_L3', 'Avg_Goals Scored_L3', 'Avg_Assists_L3', 'Avg_Goals Conceded_L3', 'Avg_ICT Index_L3', 'Avg_Threat_L3', 'Avg_Creativity_L3', 'Avg_Influence_L3', 'Avg_Total Points_L5', 'Avg_Minutes Played_L5', 'Avg_Goals Scored_L5', 'Avg_Assists_L5', 'Avg_Goals Conceded_L5', 'Avg_ICT Index_L5', 'Avg_Threat_L5', 'Avg_Creativity_L5', 'Avg_Influenc

In [6]:
print("🔍 Checking duplicates (season + GW + Player UUID)...")

dups = df_merged[df_merged.duplicated(
    subset=["Player UUID", "season", "Gameweek"],
    keep=False
)]

print(f"Found {len(dups)} duplicated rows")
display(dups.head(20))


🔍 Checking duplicates (season + GW + Player UUID)...
Found 35925 duplicated rows


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
65,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Brighton,2020-21,26,60,0,0,...,0.000000,0.8,15.8,0.0,0.0,0.2,0.72,6.6,0.70,0.20
66,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Everton,2020-21,26,60,0,0,...,0.333333,1.0,25.6,0.0,0.0,0.4,1.14,10.8,0.52,0.40
98,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Brighton,2021-22,22,0,0,0,...,0.000000,0.6,19.0,0.0,0.0,0.2,0.50,2.4,2.34,0.44
99,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Man City,2021-22,22,0,0,0,...,0.000000,0.4,12.0,0.0,0.0,0.2,0.04,0.4,0.26,0.00
100,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Brighton,2021-22,22,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
102,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Brighton,2021-22,25,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
103,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Southampton,2021-22,25,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
104,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Brighton,2021-22,25,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
105,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Brighton,2021-22,26,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
106,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,72,Aaron Connolly,Connolly,Spurs,2021-22,26,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00


In [ ]:
df_merged = df_merged.drop_duplicates(
    subset=["Player UUID", "season", "Gameweek"],
    keep="first"
).reset_index(drop=True)

print("✔️ Duplicates removed. New shape:", df_merged.shape)


In [8]:
dups.groupby(["Player UUID","season","Gameweek"]).size().sort_values(ascending=False).head(20)




Player UUID                           season   Gameweek
3e20649c-3f65-4d75-998a-a137be556e95  2021-22  36          8
                                      2020-21  26          8
                                      2021-22  29          7
9d4bab60-9257-4630-ae3b-4aabe590c687  2020-21  35          6
ae7acf9a-5a4b-4f6c-9996-752ba24ba5ca  2020-21  35          6
de21d9ed-2379-46ef-8a6d-583bce9c3ac0  2020-21  35          6
e5ccd21a-9043-43c3-9e08-9bfdb99278fd  2020-21  35          6
cf4a10bd-c897-4123-baa1-be57e82b399d  2020-21  35          6
e804ddff-59f1-4d8b-9180-7aa99075be0c  2020-21  35          6
7e39c219-b939-4251-9f05-4b46b6a1301e  2020-21  35          6
ff2ab331-cd3b-4f11-9586-5ca8af8f236a  2020-21  35          6
28ef1d1d-5b17-478a-9e9e-10a9f2b9f538  2020-21  35          6
d8c78fd7-0b64-4344-9592-c9a85ff940c1  2020-21  35          6
3e20649c-3f65-4d75-998a-a137be556e95  2021-22  26          6
a0136567-ef44-486e-899e-6e03ff50f83e  2020-21  35          6
fe157854-5ca4-48a2-a783-1bfa3

In [9]:
df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

dups = df[df.duplicated(["Player UUID", "season", "Gameweek"], keep=False)]
dups.sort_values(["Player UUID","season","Gameweek"]).head(200)


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
58674,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,597,Filip Benkovic,Benkovic,Leicester,2019-20,24,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58675,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,597,Filip Benkovic,Benkovic,Liverpool,2019-20,24,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58698,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,237,Filip Benkovic,Benkovic,Leicester,2020-21,19,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58699,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,237,Filip Benkovic,Benkovic,Fulham,2020-21,19,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58700,00015d7a-290b-46d1-bd8a-f9dcf063f9ef,237,Filip Benkovic,Benkovic,Leicester,2020-21,19,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43938,014d7dd3-095d-453e-b5dd-c7e89a775228,512,Davinson Sánchez,Sánchez,Spurs,2023-24,37,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43939,014d7dd3-095d-453e-b5dd-c7e89a775228,512,Davinson Sánchez,Sánchez,Fulham,2023-24,37,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
43940,014d7dd3-095d-453e-b5dd-c7e89a775228,512,Davinson Sánchez,Sánchez,Spurs,2023-24,37,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47928,014ea39b-97eb-4a62-9c93-443ef793bfa5,697,Dominic Sadi,Sadi,Bournemouth,2022-23,29,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
import pandas as pd

df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

# Filter Zidane Iqbal 2022-23
sub = df[
    (df["Player Name"].str.contains("Zidane", case=False, na=False)) &
    (df["season"] == "2022-23")
]

sub.to_csv("output/iqbal_2022_23.csv", index=False, encoding="utf-8-sig")
print(sub)


                                 Player UUID  Code   Player Name Web Name  \
186672  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186673  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186674  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186675  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186676  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186677  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186678  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186679  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186680  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186681  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186682  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   
186683  c10c39c3-d1a2-4c97-9bab-869dbdc207c2   551  Zidane Iqbal    Iqbal   

In [12]:
print("🧹 Fixing duplicate rows (Player UUID + season + GW)...")

df = pd.read_csv("output/training_data_v3.csv", encoding="utf-8-sig")

# Step 1 – Προτεραιότητα στα πραγματικά ματς (Minutes > 0)
df["PlayedFlag"] = df["Minutes Played"] > 0

# Step 2 – Βαθμολογούμε rows ώστε να ξέρουμε ποιο να κρατήσουμε
df["keep_rank"] = df.groupby(
    ["Player UUID", "season", "Gameweek"]
)["PlayedFlag"].transform(lambda s: s.rank(method="first", ascending=False))

# Step 3 – Κρατάμε ΜΟΝΟ το 1 καλύτερο
df_dedup = df[df["keep_rank"] == 1].drop(columns=["PlayedFlag", "keep_rank"])

print("✔️ Before:", len(df))
print("✔️ After :", len(df_dedup))
print("✔️ Removed duplicates:", len(df) - len(df_dedup))

df_dedup.to_csv("output/training_data_v3_clean.csv", index=False, encoding="utf-8-sig")
print("💾 Saved cleaned → training_data_v3_clean.csv")


🧹 Fixing duplicate rows (Player UUID + season + GW)...
✔️ Before: 194781
✔️ After : 171603
✔️ Removed duplicates: 23178
💾 Saved cleaned → training_data_v3_clean.csv


In [14]:
# ================================================================
# FINAL SAFE VALIDATION CELL
# ================================================================

import pandas as pd

print("📂 Loading cleaned training dataset...")
df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")
print(f"   → Rows: {len(df):,}\n")

# ================================================================
# 1️⃣ DUPLICATES CHECK
# ================================================================
print("🔍 Checking duplicates (Player UUID + season + GW)...")

dupes = df.duplicated(subset=["Player UUID","season","Gameweek"], keep=False)
dupe_rows = df[dupes]

print(f"   → Duplicate rows found: {len(dupe_rows):,}")
if len(dupe_rows) > 0:
    display(dupe_rows.head(20))
else:
    print("   ✔ No duplicates detected.")

print("\n" + "="*70 + "\n")

# ================================================================
# 2️⃣ MISSING OPPONENT DIFFICULTY
# ================================================================
print("🔍 Checking missing Opponent Difficulty...")
missing_od = df[df["Opponent Difficulty"].isna()]
print(f"   → Missing OD count: {len(missing_od):,}")

if len(missing_od) > 0:
    display(missing_od.head(20))
else:
    print("   ✔ No missing OD values.")

print("\n" + "="*70 + "\n")

# ================================================================
# 3️⃣ GAMEWEEK GAPS (NORMAL BEHAVIOR)
# ================================================================
print("🔍 Checking gameweek gaps per player (expected for most players)...")

gap_examples = []

for (pid, season), sub in df.groupby(["Player UUID","season"]):
    gws = sorted(sub["Gameweek"].unique())
    expected = list(range(min(gws), max(gws)+1))
    if gws != expected:
        gap_examples.append((pid, season, gws))
        
print(f"   → Players with non-continuous GW sequences: {len(gap_examples):,}")
print("   (This is normal: players skip matches)")

print("\nShowing first 5 examples:")
for g in gap_examples[:5]:
    print(g)

print("\n" + "="*70 + "\n")

# ================================================================
# 4️⃣ TEAM NAME CONSISTENCY CHECK
# ================================================================
print("🔍 Checking team name consistency using teams.csv...")

teams_df = pd.read_csv("data/2022-23/teams.csv", encoding="utf-8-sig")[["name"]]
unique_team_names = sorted(teams_df["name"].unique())

invalid_teams = df[~df["Player Team Name"].isin(unique_team_names)]

print(f"   → Invalid team names: {len(invalid_teams):,}")

if len(invalid_teams) > 0:
    display(invalid_teams.head(20))
else:
    print("   ✔ All team names match known Premier League teams.")

print("\n" + "="*70 + "\n")

# ================================================================
# 5️⃣ OPPONENT NAME CONSISTENCY
# ================================================================
print("🔍 Checking opponent name consistency...")

invalid_opps = df[~df["Opponent Name"].isin(unique_team_names)]

print(f"   → Invalid opponent names: {len(invalid_opps):,}")

if len(invalid_opps) > 0:
    display(invalid_opps.head(20))
else:
    print("   ✔ All opponent names valid.")

print("\n🎉 VALIDATION COMPLETE\n")


📂 Loading cleaned training dataset...
   → Rows: 171,603

🔍 Checking duplicates (Player UUID + season + GW)...
   → Duplicate rows found: 0
   ✔ No duplicates detected.


🔍 Checking missing Opponent Difficulty...
   → Missing OD count: 0
   ✔ No missing OD values.


🔍 Checking gameweek gaps per player (expected for most players)...
   → Players with non-continuous GW sequences: 3,474
   (This is normal: players skip matches)

Showing first 5 examples:
('00015d7a-290b-46d1-bd8a-f9dcf063f9ef', '2020-21', [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(37), np.int64(38)])
('0014c1f

,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
512,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,1,90,0,0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.00,0.00,0.000000,0.000000
513,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,2,90,0,0,...,10.000000,3.000000,90.0,0.0,0.0,0.000000,3.90,17.00,12.300000,10.000000
514,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,3,90,0,0,...,7.800000,2.500000,90.0,0.0,0.0,1.500000,2.65,11.50,7.300000,7.800000
515,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,4,90,0,0,...,6.466667,2.333333,90.0,0.0,0.0,2.333333,2.40,9.00,8.666667,6.466667
516,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,5,90,0,0,...,3.133333,2.250000,90.0,0.0,0.0,2.250000,2.00,8.25,7.125000,4.850000
517,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,6,90,1,1,...,4.000000,2.200000,90.0,0.0,0.0,2.000000,1.82,7.00,5.940000,5.520000
518,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,7,90,0,0,...,23.333333,4.200000,90.0,0.2,0.2,2.000000,4.00,15.00,9.220000,15.880000
519,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,8,90,0,0,...,25.600000,4.000000,90.0,0.2,0.2,1.600000,3.88,13.80,9.060000,16.120000
520,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,9,68,0,0,...,27.733333,4.000000,90.0,0.2,0.2,1.000000,4.30,13.80,11.040000,18.280000
521,45bae922-b93e-4c3f-8428-5ef6392406fb,83,Aaron Lennon,Lennon,Burnley,2018-19,10,0,0,0,...,7.133333,4.000000,85.6,0.2,0.2,1.200000,4.28,13.80,10.900000,18.280000




🔍 Checking opponent name consistency...
   → Invalid opponent names: 20,702


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L3,Avg_Total Points_L5,Avg_Minutes Played_L5,Avg_Goals Scored_L5,Avg_Assists_L5,Avg_Goals Conceded_L5,Avg_ICT Index_L5,Avg_Threat_L5,Avg_Creativity_L5,Avg_Influence_L5
7,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,32,0,0,0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00
15,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,534,Aaron Connolly,Connolly,Brighton,2019-20,5,6,0,0,...,0.200000,1.0,24.0,0.0,0.0,1.0,0.00,0.0,0.10,0.20
21,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,534,Aaron Connolly,Connolly,Brighton,2019-20,11,85,0,0,...,24.866667,4.8,49.6,0.4,0.4,0.8,5.30,29.6,8.02,15.36
28,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,534,Aaron Connolly,Connolly,Brighton,2019-20,18,45,0,0,...,2.066667,0.8,30.0,0.0,0.0,0.6,1.08,8.8,0.40,1.48
36,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,534,Aaron Connolly,Connolly,Brighton,2019-20,26,0,0,0,...,0.000000,0.8,28.4,0.0,0.0,0.2,1.42,11.4,2.20,0.68
37,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,534,Aaron Connolly,Connolly,Brighton,2019-20,27,15,0,0,...,0.000000,0.6,19.4,0.0,0.0,0.2,0.76,5.6,2.08,0.00
45,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Brighton,2020-21,6,0,0,0,...,6.733333,3.4,56.8,0.2,0.2,1.4,3.18,13.8,6.86,11.24
47,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Brighton,2020-21,8,4,0,0,...,5.733333,1.6,30.0,0.0,0.2,1.0,1.12,2.8,4.54,4.04
53,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Brighton,2020-21,14,90,0,0,...,1.133333,1.0,30.6,0.0,0.0,0.4,1.60,12.8,2.62,0.68
62,f6ebd126-7139-4c02-9071-fe0c85cb5e2d,78,Aaron Connolly,Connolly,Brighton,2020-21,23,61,0,0,...,0.333333,0.4,3.6,0.0,0.0,0.0,0.42,3.6,0.38,0.20



🎉 VALIDATION COMPLETE



In [15]:
# ============================================================
# ADD TEAM CONTRIBUTION FEATURES (NO LEAKAGE) TO v3_clean
# ============================================================

print("\n📊 Adding Team Contribution Features (NO LEAKAGE)...")

import numpy as np
import pandas as pd

df = pd.read_csv("output/training_data_v3_clean.csv", encoding="utf-8-sig")

# Ensure numeric types
numeric_cols = [
    "Total Points", "Minutes Played", "Goals Scored", "Assists",
    "Clean Sheet", "Goals Conceded", "Yellow Card", "Red Cards",
    "Threat", "ICT Index", "Influence", "Creativity"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# ============================================================
# STEP 1 — TEAM TOTAL POINTS PER GAMEWEEK
# ============================================================
team_points = (
    df.groupby(["Player Team Name", "season", "Gameweek"])["Total Points"]
      .sum()
      .reset_index()
      .rename(columns={"Total Points": "Team_Total_Points_GW"})
)

team_points["Team_Total_Points_GW"] = pd.to_numeric(
    team_points["Team_Total_Points_GW"], errors="coerce"
).fillna(0)

# Cumulative team points (per season)
team_points["Team_Total_Points_CUM"] = (
    team_points.groupby(["Player Team Name", "season"])["Team_Total_Points_GW"]
               .cumsum()
)

# Lagged cumulative (causal)
team_points["Team_Total_Points"] = (
    team_points.groupby(["Player Team Name", "season"])["Team_Total_Points_CUM"]
               .shift(1)
)

# Merge back into main DF
df = df.merge(
    team_points[["Player Team Name", "season", "Gameweek",
                 "Team_Total_Points_GW", "Team_Total_Points"]],
    on=["Player Team Name", "season", "Gameweek"],
    how="left"
)

df["Team_Total_Points"].fillna(0, inplace=True)

# ============================================================
# STEP 2 — PLAYER CUMULATIVE POINTS (CAUSAL)
# ============================================================
df["Player_Season_Points"] = (
    df.groupby(["Player UUID", "season"])["Total Points"]
      .cumsum()
      .shift(1)
      .fillna(0)
)

# ============================================================
# STEP 3 — TEAM CONTRIBUTION (GW + CAUSAL)
# ============================================================
df["Team_Points_Contribution_GW"] = np.where(
    df["Team_Total_Points_GW"] > 0,
    df["Total Points"] / df["Team_Total_Points_GW"],
    0
).round(6)

df["Team_Points_Contribution_Causal"] = np.where(
    df["Team_Total_Points"] > 0,
    df["Player_Season_Points"] / df["Team_Total_Points"],
    0
).round(6)

# ============================================================
# STEP 4 — % PERCENTAGES
# ============================================================
df["Team_Points_Contribution_GW_Pct"] = (df["Team_Points_Contribution_GW"] * 100).round(2)
df["Team_Points_Contribution_Causal_Pct"] = (df["Team_Points_Contribution_Causal"] * 100).round(2)

# ============================================================
# STEP 5 — RANKS
# ============================================================
df["Team_Contribution_Rank_GW"] = (
    df.groupby(["Player Team Name", "season", "Gameweek"])["Team_Points_Contribution_GW"]
      .rank(method="min", ascending=False, pct=True)
)

df["Team_Contribution_Rank_Causal"] = (
    df.groupby(["Player Team Name", "season", "Gameweek"])["Team_Points_Contribution_Causal"]
      .rank(method="min", ascending=False, pct=True)
)

# ============================================================
# SAVE
# ============================================================
df.to_csv("output/training_data_v4_with_contrib.csv", index=False, encoding="utf-8-sig")
print("🎉 DONE! Saved → training_data_v4.csv")

print("\n📌 Preview:")
display(df.head())



📊 Adding Team Contribution Features (NO LEAKAGE)...


C:\Users\SOFI\AppData\Local\Temp\ipykernel_20576\1474510821.py:56: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Team_Total_Points"].fillna(0, inplace=True)


🎉 DONE! Saved → training_data_v4.csv

📌 Preview:


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L5,Team_Total_Points_GW,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW,Team_Points_Contribution_Causal,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct,Team_Contribution_Rank_GW,Team_Contribution_Rank_Causal
0,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,25,0,0,0,...,0.0,20,1068.0,0.0,0.0,0.0,0.0,0.0,0.363636,0.636364
1,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,26,0,0,0,...,0.0,26,1088.0,0.0,0.0,0.0,0.0,0.0,0.326087,0.608696
2,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,27,0,0,0,...,0.0,104,1114.0,0.0,0.0,0.0,0.0,0.0,0.391304,0.608696
3,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,28,0,0,0,...,0.0,74,1218.0,0.0,0.0,0.0,0.0,0.0,0.319149,0.638298
4,16b72858-75e4-4125-ad3b-e13f81e6d815,774,Aaron Anselmino,Anselmino,Chelsea,2024-25,29,0,0,0,...,0.0,25,1292.0,0.0,0.0,0.0,0.0,0.0,0.361702,0.638298


In [16]:
# ============================================================
# SANITY CHECK FOR TEAM CONTRIBUTION FEATURES
# ============================================================

import numpy as np
import pandas as pd

df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig")

print("📂 Loaded:", len(df), "rows")

required_cols = [
    "Team_Total_Points_GW", "Team_Total_Points", "Player_Season_Points",
    "Team_Points_Contribution_GW", "Team_Points_Contribution_Causal",
    "Team_Points_Contribution_GW_Pct", "Team_Points_Contribution_Causal_Pct",
    "Team_Contribution_Rank_GW", "Team_Contribution_Rank_Causal"
]

print("\n📝 Checking required columns...")
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print("❌ Missing columns:", missing)
else:
    print("✔ All contribution columns present.")

# ------------------------------------------------------------
# Check for NaNs
# ------------------------------------------------------------
print("\n🔍 Checking NaNs in contribution cols...")
nan_counts = df[required_cols].isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else "✔ No NaNs")

# ------------------------------------------------------------
# Check GW1 causal logic
# ------------------------------------------------------------
print("\n📊 Checking Gameweek 1 causal values...")

gw1 = df[df["Gameweek"] == 1]

print("Team_Total_Points > 0 on GW1:", (gw1["Team_Total_Points"] > 0).sum())
print("Player_Season_Points > 0 on GW1:", (gw1["Player_Season_Points"] > 0).sum())

# ------------------------------------------------------------
# Check sum of GW contributions ≈ 1
# ------------------------------------------------------------
print("\n📏 Checking if GW contributions sum to ~1 per team/week...")

group_sum = df.groupby(["Player Team Name", "season", "Gameweek"])["Team_Points_Contribution_GW"].sum()
bad_groups = group_sum[(group_sum < 0.999) | (group_sum > 1.001)]

print("Groups failing contribution sum check:", len(bad_groups))

if len(bad_groups) > 0:
    print(bad_groups.head())

# ------------------------------------------------------------
# Check ranks are within valid range
# ------------------------------------------------------------
print("\n🏅 Checking rank validity...")

rank_cols = ["Team_Contribution_Rank_GW", "Team_Contribution_Rank_Causal"]

for col in rank_cols:
    bad = df[(df[col] < 0) | (df[col] > 1)]
    print(f"{col}: invalid values:", len(bad))

print("\n📊 Rank samples:")
display(df[["Team_Points_Contribution_GW", "Team_Contribution_Rank_GW"]].head())

# ------------------------------------------------------------
# Summary stats
# ------------------------------------------------------------
print("\n📈 Contribution summary:")
display(df[["Team_Points_Contribution_GW", "Team_Points_Contribution_Causal"]].describe())

print("\n🎯 Top 10 GW contributions:")
display(df.sort_values("Team_Points_Contribution_GW", ascending=False).head(10))


📂 Loaded: 171603 rows

📝 Checking required columns...
✔ All contribution columns present.

🔍 Checking NaNs in contribution cols...
✔ No NaNs

📊 Checking Gameweek 1 causal values...
Team_Total_Points > 0 on GW1: 0
Player_Season_Points > 0 on GW1: 3401

📏 Checking if GW contributions sum to ~1 per team/week...
Groups failing contribution sum check: 4
Player Team Name  season   Gameweek
Norwich           2021-22  9           0.0
Southampton       2019-20  10          0.0
                  2020-21  22          0.0
Watford           2019-20  6           0.0
Name: Team_Points_Contribution_GW, dtype: float64

🏅 Checking rank validity...
Team_Contribution_Rank_GW: invalid values: 0
Team_Contribution_Rank_Causal: invalid values: 0

📊 Rank samples:


,Team_Points_Contribution_GW,Team_Contribution_Rank_GW
0,0.0,0.363636
1,0.0,0.326087
2,0.0,0.391304
3,0.0,0.319149
4,0.0,0.361702



📈 Contribution summary:


,Team_Points_Contribution_GW,Team_Points_Contribution_Causal
count,171603.000000,171603.000000
mean,0.029918,0.029778
std,0.054894,0.053246
min,-1.500000,-0.125000
25%,0.000000,0.000000
50%,0.000000,0.014245
75%,0.045455,0.051282
max,1.000000,5.947368



🎯 Top 10 GW contributions:


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L5,Team_Total_Points_GW,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW,Team_Points_Contribution_Causal,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct,Team_Contribution_Rank_GW,Team_Contribution_Rank_Causal
122245,5848ebf5-e75d-4589-88af-643c1e5bc3ed,691,Norberto Bercique Gomes Betuncal,Beto,Everton,2023-24,33,90,0,0,...,18.52,2,1180.0,45.0,1.000000,0.038136,100.00,3.81,0.030303,0.363636
69436,5c9e13f5-0f64-4860-b596-6cf27be153f9,360,James McAtee,McAtee,Sheffield Utd,2023-24,6,74,0,0,...,1.60,2,136.0,4.0,1.000000,0.029412,100.00,2.94,0.029412,0.323529
24648,8ee9a9b8-cbe3-4d8a-b3de-bfc65dbd0c8c,33,Cameron Archer,Archer,Sheffield Utd,2023-24,6,90,0,0,...,7.72,2,136.0,13.0,1.000000,0.095588,100.00,9.56,0.029412,0.088235
1552,39f2024b-3ebb-4f11-a088-4541e6f8437e,249,Abdoulaye Doucouré,A.Doucoure,Everton,2023-24,33,90,0,0,...,5.12,2,1180.0,93.0,1.000000,0.078814,100.00,7.88,0.030303,0.090909
123573,35fd02fc-beca-437b-b47d-061e91f1b12f,488,Oliver Norwood,Norwood,Sheffield Utd,2023-24,6,61,0,0,...,13.24,2,136.0,8.0,1.000000,0.058824,100.00,5.88,0.029412,0.147059
157138,db6f72e5-019c-4f68-830d-48f813cdf0f7,638,Vini de Souza Costa,Vini Souza,Sheffield Utd,2023-24,6,90,0,0,...,11.76,2,136.0,7.0,1.000000,0.051471,100.00,5.15,0.029412,0.205882
43001,a327996d-9743-4091-8be0-592291cdf4bc,259,Dwight McNeil,McNeil,Everton,2023-24,33,90,0,0,...,12.36,2,1180.0,93.0,1.000000,0.078814,100.00,7.88,0.030303,0.090909
56624,818df6c7-b1dd-4acd-aa01-a0e7037344e6,663,Gustavo Hamer,Hamer,Sheffield Utd,2023-24,6,65,0,0,...,28.20,2,136.0,20.0,1.000000,0.147059,100.00,14.71,0.029412,0.029412
70808,48a04abc-eb8c-4fb1-b130-981660896b8d,664,James Ward-Prowse,Ward-Prowse,West Ham,2023-24,24,90,0,0,...,32.44,3,954.0,103.0,0.666667,0.107966,66.67,10.80,0.030303,0.060606
55155,5cca0a97-0c3c-4330-aea4-9b94e5226c9c,618,Georginio Rutter,Georginio,Brighton,2024-25,24,90,0,0,...,16.88,3,956.0,70.0,0.666667,0.073222,66.67,7.32,0.020408,0.081633


In [17]:
import pandas as pd

df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig")

print("📂 Loaded:", len(df), "rows")

# -----------------------------------------------
# 1. Πόσα κενά υπάρχουν στο Position;
# -----------------------------------------------
missing_pos = df["Position"].isna().sum()
print(f"\n🔍 Missing Position values: {missing_pos:,}")

# -----------------------------------------------
# 2. Δείξε κάποιες γραμμές με missing position
# -----------------------------------------------
print("\n🧪 Sample rows with missing Position:")
display(df[df["Position"].isna()].head(20))

# -----------------------------------------------
# 3. Διάγραμμα συχνότητας ανά σεζόν
# -----------------------------------------------
print("\n📊 Missing positions per season:")
display(df[df["Position"].isna()]
        .groupby("season")["Player Name"]
        .count()
        .sort_index())

# -----------------------------------------------
# 4. Διάγραμμα συχνότητας ανά ομάδα
# -----------------------------------------------
print("\n📊 Missing positions per TEAM:")
display(df[df["Position"].isna()]
        .groupby("Player Team Name")["Player Name"]
        .count()
        .sort_values(ascending=False)
        .head(20))

# -----------------------------------------------
# 5. Τι element_type είχαν αυτοί οι παίκτες (αν υπάρχει)
# -----------------------------------------------
if "element_type" in df.columns:
    print("\n📊 element_type breakdown for missing Position:")
    display(
        df[df["Position"].isna()]
        .groupby("element_type")["Player Name"]
        .count()
    )
else:
    print("\n⚠️ No element_type column in the CSV to debug positions.")


📂 Loaded: 171603 rows

🔍 Missing Position values: 312

🧪 Sample rows with missing Position:


,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,Assists,...,Avg_Influence_L5,Team_Total_Points_GW,Team_Total_Points,Player_Season_Points,Team_Points_Contribution_GW,Team_Points_Contribution_Causal,Team_Points_Contribution_GW_Pct,Team_Points_Contribution_Causal_Pct,Team_Contribution_Rank_GW,Team_Contribution_Rank_Causal
8874,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,23,0,0,0,...,0.0,106,990.0,0.0,0.122642,0.000000,12.26,0.00,0.048780,0.658537
8875,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,24,0,0,0,...,0.0,20,1096.0,13.0,0.000000,0.011861,0.00,1.19,0.333333,0.452381
8876,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,25,0,0,0,...,0.0,61,1116.0,13.0,0.147541,0.011649,14.75,1.16,0.048780,0.463415
8877,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,26,0,0,0,...,0.0,18,1177.0,22.0,0.000000,0.018692,0.00,1.87,0.341463,0.390244
8878,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,27,0,0,0,...,0.0,32,1195.0,22.0,0.031250,0.018410,3.12,1.84,0.170732,0.390244
8879,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,28,0,0,0,...,0.0,42,1227.0,23.0,0.119048,0.018745,11.90,1.87,0.073171,0.390244
8880,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,29,0,0,0,...,0.0,24,1269.0,28.0,0.041667,0.022065,4.17,2.21,0.170732,0.390244
8881,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,30,0,0,0,...,0.0,29,1293.0,29.0,0.034483,0.022428,3.45,2.24,0.121951,0.390244
8882,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,31,0,0,0,...,0.0,44,1322.0,30.0,0.113636,0.022693,11.36,2.27,0.073171,0.390244
8883,5d064b95-d989-43d1-a43b-80bc7f325c38,738,Andoni Iraola,Iraola,Bournemouth,2024-25,32,0,0,0,...,0.0,64,1366.0,35.0,0.140625,0.025622,14.06,2.56,0.073171,0.365854



📊 Missing positions per season:


season
2024-25    312
Name: Player Name, dtype: int64


📊 Missing positions per TEAM:


Player Team Name
Brentford         16
Bournemouth       16
Chelsea           16
Brighton          16
West Ham          16
Wolves            16
Everton           16
Fulham            16
Leicester         16
Ipswich           16
Nott'm Forest     16
Man Utd           16
Spurs             16
Southampton       16
Newcastle         15
Arsenal           15
Man City          15
Liverpool         15
Aston Villa       14
Crystal Palace    14
Name: Player Name, dtype: int64


⚠️ No element_type column in the CSV to debug positions.


In [20]:
import pandas as pd
from pathlib import Path

print("📂 Loading training_data_v4_with_contrib.csv ...")
df = pd.read_csv("output/training_data_v4_with_contrib.csv", encoding="utf-8-sig")
print("   → Rows:", len(df))

# ------------------------------------------------------------
# 1️⃣ Load players_raw for ALL seasons
# ------------------------------------------------------------

DATA_ROOT = Path("data")

players_raw_list = []

for season_folder in sorted(DATA_ROOT.iterdir()):
    pr_path = season_folder / "players_raw.csv"
    if pr_path.exists():
        print(f"📥 Loading → {pr_path}")
        temp = pd.read_csv(pr_path)

        # normalize ID column
        if "element" in temp.columns:
            temp = temp.rename(columns={"element": "Code"})
        elif "id" in temp.columns:
            temp = temp.rename(columns={"id": "Code"})
        else:
            continue

        if "element_type" not in temp.columns:
            print("⚠ Missing element_type → skipping")
            continue

        temp = temp[["Code", "element_type"]]
        temp["Code"] = pd.to_numeric(temp["Code"], errors="coerce")
        players_raw_list.append(temp)

players_raw = pd.concat(players_raw_list, ignore_index=True).drop_duplicates("Code")

print("✔ Total players_raw combined:", len(players_raw))

# ------------------------------------------------------------
# 2️⃣ Merge positions back into training dataset
# ------------------------------------------------------------

df["Code"] = pd.to_numeric(df["Code"], errors="coerce")

df_fix = df.merge(players_raw, on="Code", how="left")

POS_MAP = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
df_fix["Position"] = df_fix["element_type"].map(POS_MAP)

missing_after = df_fix["Position"].isna().sum()
print("\n🔍 Missing positions AFTER FIX:", missing_after)

# ------------------------------------------------------------
# 3️⃣ Save final fixed version
# ------------------------------------------------------------
out_path = "output/training_data_v6_positions_fixed.csv"
df_fix.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Saved → {out_path}")


📂 Loading training_data_v4_with_contrib.csv ...
   → Rows: 171603
📥 Loading → data\2018-19\players_raw.csv
📥 Loading → data\2019-20\players_raw.csv
📥 Loading → data\2020-21\players_raw.csv
📥 Loading → data\2021-22\players_raw.csv
📥 Loading → data\2022-23\players_raw.csv
📥 Loading → data\2023-24\players_raw.csv
📥 Loading → data\2024-25\players_raw.csv
📥 Loading → data\2025-26\players_raw.csv
✔ Total players_raw combined: 866

🔍 Missing positions AFTER FIX: 0

💾 Saved → output/training_data_v6_positions_fixed.csv


In [1]:
# CLEANING: EXTRACT & REMOVE 0-MINUTE ROWS FROM TRAINING DATASET 
# FOR LINEAR REGRESSION PURPOSES

import pandas as pd
from pathlib import Path


# PATHS

INPUT_PATH = Path("output/training_data_v6_positions_fixed.csv")
ZERO_OUT = Path("output/players_0minutes.csv")
CLEAN_OUT = Path("output/training_data_v6_no0min.csv")

print("📂 Loading dataset...")
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
print(f"   → Rows loaded: {len(df):,}")


#  EXTRACT ZERO-MINUTES ROWS

zero_df = df[df["Minutes Played"] == 0].copy()

print(f"🔍 Found players with 0 minutes: {len(zero_df):,}")

# Save zero-minutes rows
zero_df.to_csv(ZERO_OUT, index=False, encoding="utf-8-sig")
print(f"💾 Saved zero-minute rows → {ZERO_OUT}")


#  REMOVE ZERO-MINUTE ROWS FROM MAIN DATASET

df_clean = df[df["Minutes Played"] != 0].copy()

print(f"🧽 Cleaned dataset rows: {len(df_clean):,}")
print(f"🗑 Removed: {len(df) - len(df_clean):,} rows")


#  SAVE CLEANED DATASET

df_clean.to_csv(CLEAN_OUT, index=False, encoding="utf-8-sig")
print(f"💾 Saved cleaned dataset → {CLEAN_OUT}")

print("\n🎉 DONE! Zero-minute rows extracted & removed successfully.")


📂 Loading dataset...
   → Rows loaded: 171,603
🔍 Found players with 0 minutes: 97,568
💾 Saved zero-minute rows → output\players_0minutes.csv
🧽 Cleaned dataset rows: 74,035
🗑 Removed: 97,568 rows
💾 Saved cleaned dataset → output\training_data_v6_no0min.csv

🎉 DONE! Zero-minute rows extracted & removed successfully.


In [2]:
# CREATE A TRIAL DATASET WITH SELECTED SEASONS ONLY


# PATHS

INPUT_PATH = Path("output/training_data_v6_no0min.csv")
TRIAL_OUT = Path("output/training_data_v6_no0min_trial.csv")

print("📂 Loading dataset...")
df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
print(f"   → Rows loaded: {len(df):,}")


# KEEP ONLY SELECTED SEASONS

keep_seasons = ["2023-24", "2024-25", "2025-26"]

df_trial = df[df["season"].isin(keep_seasons)].copy()

print(f"🧪 Trial dataset rows after filtering: {len(df_trial):,}")
print(f"🗑 Removed rows: {len(df) - len(df_trial):,}")


#  SAVE TRIAL DATASET

df_trial.to_csv(TRIAL_OUT, index=False, encoding="utf-8-sig")
print(f"💾 Saved trial dataset → {TRIAL_OUT}")

print("\n🎉 DONE! training_data_v6_no0min_trial.csv created successfully.")


📂 Loading dataset...
   → Rows loaded: 74,035
🧪 Trial dataset rows after filtering: 25,735
🗑 Removed rows: 48,300
💾 Saved trial dataset → output\training_data_v6_no0min_trial.csv

🎉 DONE! training_data_v6_no0min_trial.csv created successfully.
